# Z-Scores & Standardization
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** a z-score from a raw value, a population mean, and a standard deviation
- **Interpret** a z-score as the number of standard deviations a value sits above or below the mean
- **Explain** why standardization is essential for comparing values across different scales or units

> **Tip:** Start by moving the **raw value slider** and watch the z-score update, pay attention to the moment it crosses ±2, which marks the boundary of the most common 95% of values.

---
## How we got here

In *12: The Normal Distribution* we learned about the 68–95–99.7 rule: fixed fractions of data always fall within 1, 2, and 3 standard deviations of the mean. In *14: Sampling Distributions* we tracked how sample statistics vary. The z-score is the tool that converts any value into this universal standard: how many σ above or below the mean is this point? Once in z-score form, every normal distribution becomes comparable.

---
## Why this matters for data science

Standardization (converting to z-scores) is one of the most common preprocessing steps in machine learning. Algorithms like k-nearest neighbors, support vector machines, and neural networks are sensitive to feature scale, a feature measured in thousands of dollars will dominate one measured in percentages unless both are standardized first. `sklearn.preprocessing.StandardScaler` computes exactly the z-score transformation you will learn here.

---
## Try it yourself

In [2]:
from tkh_utils import make_slider, make_output, display_widget, register_observer

MU, SIGMA = 70.0, 10.0
_x_range = np.linspace(MU - 4 * SIGMA, MU + 4 * SIGMA, 500)

out = make_output()
x_slider = make_slider(40.0, 100.0, 0.5, 70.0, "Exam score (x):")

def render(x_val):
    z = (x_val - MU) / SIGMA
    p = 1 - stats.norm.cdf(x_val, MU, SIGMA)
    y_curve = stats.norm.pdf(_x_range, MU, SIGMA)
    x_shade = _x_range[_x_range >= x_val]
    y_shade = stats.norm.pdf(x_shade, MU, SIGMA)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=_x_range, y=y_curve,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name=f"N(μ={MU:.0f}, σ={SIGMA:.0f})",
    ))
    if len(x_shade):
        fig.add_trace(go.Scatter(
            x=np.concatenate([[x_shade[0]], x_shade, [x_shade[-1]]]),
            y=np.concatenate([[0], y_shade, [0]]),
            fill="toself",
            fillcolor="rgba(247,103,7,0.30)",
            line=dict(color=PALETTE["secondary"], width=0),
            name=f"P(X > x) = {p:.4f}",
        ))
    fig.add_vline(
        x=x_val,
        line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
        annotation_text=f"x = {x_val:.1f}",
    )
    layout = base_layout(
        title=f"x = {x_val:.1f}  →  z = {z:.2f}  |  P(X > x) = {p:.4f}  |  μ = {MU:.0f}, σ = {SIGMA:.0f}",
        xaxis_title="Exam Score",
        yaxis_title="Probability Density",
    )
    layout.update(xaxis=dict(range=[MU - 4 * SIGMA, MU + 4 * SIGMA]))
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer([x_slider], lambda change: render(x_slider.value))
display_widget([x_slider], out)
render(x_slider.value)


---
## What's happening?

A z-score expresses how unusual a value is relative to its distribution. It answers: "Is this value typical, or is it far out in the tail?"

| Symbol | Meaning |
|--------|---------|
| x | The raw observed value |
| μ | The population mean |
| σ | The population standard deviation |
| z | The standardized score (number of SDs from mean) |

$$z = \frac{x - \mu}{\sigma}$$

Once you compute z, you can look up the corresponding probability in the **standard normal** N(0, 1), a normal distribution with mean 0 and standard deviation 1. Every normal distribution can be converted to this single reference distribution.

### A z-score is a percentile in disguise

A z-score of +1.645 corresponds to the 95th percentile, 95% of values fall below it. A z-score of −1.96 is the 2.5th percentile. These cutoffs appear again in confidence intervals and hypothesis testing.

Return to the widget and set x to the 95th percentile of the distribution, verify the z-score reads approximately 1.645.

---
## Real-world example: Comparing SAT and ACT scores

A student scored 1320 on the SAT and 29 on the ACT. Which score is more impressive? The raw numbers are on completely different scales, the only fair comparison is via z-scores.

The chart shows where each score falls within its respective distribution (based on 2023 national statistics). Notice:

- **Notice:** Both scores sit about 0.85–0.90 standard deviations above their respective means, nearly identical relative performance despite the very different raw numbers
- **Notice:** The shaded area shows the proportion of test-takers each score outperforms, this is the percentile, directly readable from the z-score
- **Notice:** Converting to z-scores collapsed two incomparable scales into a single question: how many SDs above average is this student?

> **Discussion question:** A college requires a minimum SAT score of 1200 or ACT score of 26. Using z-scores, determine which requirement is harder to meet, and whether they are truly equivalent benchmarks.

In [3]:
# ── SAT vs ACT z-score comparison (2023 national statistics) ────────────────────
tests = {
    "SAT": {"score": 1320, "mu": 1060,  "sigma": 210,  "color": PALETTE["primary"]},
    "ACT": {"score": 29,   "mu": 20.9,  "sigma": 6.0,  "color": PALETTE["secondary"]},
}

fig = go.Figure()
for name, t in tests.items():
    z = (t["score"] - t["mu"]) / t["sigma"]
    pct = stats.norm.cdf(z) * 100

    # Standardize to z-scale for comparison
    x_z = np.linspace(-4, 4, 400)
    y_z = stats.norm.pdf(x_z)
    x_shade = x_z[x_z <= z]

    fig.add_trace(go.Scatter(
        x=x_z, y=y_z + (0.05 if name == "ACT" else 0),  # slight vertical offset for clarity
        mode="lines", line=dict(color=t["color"], width=2.5),
        name=f"{name}  z={z:.2f}  ({pct:.0f}th percentile)",
    ))

fig.add_vline(x=(1320 - 1060) / 210, line_color=PALETTE["primary"],
              line_dash="dash", annotation_text="SAT z")
fig.add_vline(x=(29 - 20.9) / 6.0, line_color=PALETTE["secondary"],
              line_dash="dot", annotation_text="ACT z")
layout = base_layout(
    title="SAT vs ACT Scores — Standardized to Z-Scale for Fair Comparison",
    xaxis_title="Z-Score (standard deviations from mean)",
    yaxis_title="Probability Density (standardized)",
)
layout.update(xaxis=dict(range=[-4, 4]))
fig.update_layout(layout)
fig.show()

### Common z-score benchmarks to memorize

| Z-score | Percentile | Meaning |
|---------|-----------|---------|
| −3.0 | 0.13% | Far below average: extremely rare |
| −2.0 | 2.28% | Bottom 2% |
| −1.0 | 15.87% | Below average |
| 0.0 | 50.00% | Exactly at the mean |
| +1.0 | 84.13% | Above average |
| +1.645 | 95.00% | Used in 90% confidence intervals |
| +1.96 | 97.72% | Used in 95% confidence intervals |
| +2.576 | 99.50% | Used in 99% confidence intervals |

---
## Key takeaway

> **A z-score converts any value to a universal scale by measuring how many standard deviations it sits from the mean — making scores on different scales directly comparable.**

---
*Next up: Confidence Intervals — using the sampling distribution and z-scores to build a range that captures the true population parameter with known probability*